# Oracle Price Integration Configuration Testing

This notebook demonstrates and tests the configuration-driven Oracle Price Integration Engine.

## Overview

The Oracle Price Integration Engine now supports:
- Configuration-driven oracle providers
- Flexible aggregation methods
- Configurable validation thresholds
- Emergency fallback mechanisms
- Price manipulation detection
- MCP service integration


In [ ]:
# Setup imports and paths
import sys
from pathlib import Path
import json
from datetime import datetime, timedelta

# Add parent directory to path
parent_dir = Path('..').resolve()
sys.path.insert(0, str(parent_dir))

print(f"Working directory: {parent_dir}")
print(f"Python path updated")

## 1. Configuration Loading and Inspection

In [ ]:
# Load and inspect configuration
from core.config import load_config, OracleEngineConfig

# Load configuration from YAML file
config = load_config()

print("Oracle Price Integration Configuration Loaded")
print("=" * 50)

print(f"\nMCP Services:")
print(f"  Algorand Reader: {config.mcp_services.algorand_reader_url}")
print(f"  Market Data: {config.mcp_services.market_data_url}")

print(f"\nOracle Providers:")
for name, provider in config.oracle_providers.items():
    print(f"  {name}:")
    print(f"    Enabled: {provider.enabled}")
    print(f"    Reliability: {provider.reliability_score}")
    print(f"    Variance: {provider.variance}")
    print(f"    Latency: {provider.latency_seconds}s")

print(f"\nValidation Settings:")
print(f"  Confidence Threshold: {config.validation.confidence_threshold}")
print(f"  Min Data Points: {config.validation.min_data_points}")
print(f"  Max Price Deviation: {config.validation.max_price_deviation_percent}%")

print(f"\nAggregation Method: {config.aggregation.method}")
print(f"Weight Factors: {config.aggregation.weight_factors}")

## 2. Oracle Manager Initialization

In [ ]:
# Initialize the oracle manager with configuration
from oracle_integration_configured import ConfigurableOracleManager

oracle_manager = ConfigurableOracleManager()

print("Oracle Manager Initialized Successfully")
print("=" * 40)

# Display oracle status
print("\nOracle Status:")
for oracle_name, status in oracle_manager.oracle_status.items():
    reliability = oracle_manager.reliability_scores.get(oracle_name, 0.0)
    print(f"  {oracle_name}: {status.value} (reliability: {reliability:.2f})")

# Get configuration summary
config_summary = oracle_manager.get_config_summary()
print(f"\nConfiguration Summary:")
print(json.dumps(config_summary, indent=2, default=str))

## 3. Price Feed Retrieval Testing

In [ ]:
# Test price feed retrieval for different assets
test_assets = ["ALGO", "USDC", "BTC", "ETH"]

print("Price Feed Retrieval Test")
print("=" * 30)

price_data = {}

for asset in test_assets:
    print(f"\n{asset} Price Feeds:")
    feeds = oracle_manager.get_price_feeds(asset)
    
    price_data[asset] = feeds
    
    if feeds:
        print(f"  Retrieved {len(feeds)} feeds")
        for feed in feeds:
            age_seconds = (datetime.now() - feed.timestamp).total_seconds()
            print(f"    {feed.oracle_name}: ${feed.price_usd:.4f} "
                  f"(confidence: {feed.confidence:.2f}, age: {age_seconds:.0f}s)")
    else:
        print(f"  No feeds available for {asset}")

## 4. Price Aggregation Testing

In [ ]:
# Test price aggregation using configuration
print("Price Aggregation Test")
print("=" * 25)

for asset, feeds in price_data.items():
    if feeds and len(feeds) > 0:
        print(f"\n{asset} Aggregation:")
        
        # Show individual prices
        print(f"  Individual Prices:")
        for feed in feeds:
            print(f"    {feed.oracle_name}: ${feed.price_usd:.4f}")
        
        # Aggregate prices
        try:
            aggregated = oracle_manager.aggregate_prices(feeds)
            
            print(f"  Aggregated Result:")
            print(f"    Consensus Price: ${aggregated.consensus_price:.4f}")
            print(f"    Price Confidence: {aggregated.price_confidence:.2f}")
            print(f"    Price Deviation: {aggregated.price_deviation:.4f}")
            print(f"    Oracle Count: {aggregated.oracle_count}")
            print(f"    Staleness Score: {aggregated.staleness_score:.2f}")
            print(f"    Consensus Strength: {aggregated.consensus_strength:.2f}")
            
        except Exception as e:
            print(f"    Error aggregating prices: {e}")

## 5. Reliable Price Retrieval with Fallbacks

In [ ]:
# Test reliable price retrieval (includes fallbacks)
print("Reliable Price Retrieval Test")
print("=" * 35)

for asset in test_assets:
    print(f"\n{asset} Reliable Price:")
    
    reliable_price = oracle_manager.get_reliable_price(asset)
    
    if reliable_price:
        print(f"  Price: ${reliable_price.consensus_price:.4f}")
        print(f"  Confidence: {reliable_price.price_confidence:.2f}")
        print(f"  Oracle Count: {reliable_price.oracle_count}")
        print(f"  Source: {[p.oracle_name for p in reliable_price.oracle_prices]}")
        
        # Check if it's a fallback price
        if any(p.oracle_name == "emergency_fallback" for p in reliable_price.oracle_prices):
            print(f"  ⚠️  Using emergency fallback price")
    else:
        print(f"  No reliable price available")

## 6. Asset-Specific Configuration Testing

In [ ]:
# Test asset-specific configurations
print("Asset-Specific Configuration Test")
print("=" * 40)

for asset in test_assets:
    refresh_interval = oracle_manager.get_asset_refresh_interval(asset)
    
    asset_config = oracle_manager.config.asset_configurations.get(asset)
    
    print(f"\n{asset} Configuration:")
    print(f"  Refresh Interval: {refresh_interval}s")
    
    if asset_config:
        print(f"  Priority: {asset_config.priority}")
        print(f"  Confidence Override: {asset_config.confidence_threshold_override}")
        print(f"  Staleness Override: {asset_config.max_staleness_override}")
    else:
        print(f"  Using default configuration")

## 7. Manipulation Detection Testing

In [ ]:
# Test price manipulation detection
from oracle_integration_configured import PriceFeed

print("Price Manipulation Detection Test")
print("=" * 40)

# Create mock price history with normal and suspicious patterns
asset_symbol = "ALGO"
base_time = datetime.now() - timedelta(hours=2)

# Normal price progression
normal_feeds = []
for i in range(15):
    feed = PriceFeed(
        oracle_name="chainlink",
        asset_symbol=asset_symbol,
        price_usd=0.25 + (i * 0.001),  # Gradual increase
        timestamp=base_time + timedelta(minutes=i * 3),
        confidence=0.95
    )
    normal_feeds.append(feed)

# Add suspicious price spike
spike_feed = PriceFeed(
    oracle_name="chainlink",
    asset_symbol=asset_symbol,
    price_usd=0.35,  # 40% spike
    timestamp=datetime.now() - timedelta(minutes=5),
    confidence=0.95
)
normal_feeds.append(spike_feed)

# Set mock price history
oracle_manager.price_history[asset_symbol] = normal_feeds

print(f"Created mock price history with {len(normal_feeds)} data points")
print(f"Price range: ${min(f.price_usd for f in normal_feeds):.4f} - ${max(f.price_usd for f in normal_feeds):.4f}")

# Run manipulation detection
manipulation_result = oracle_manager.detect_price_manipulation(asset_symbol)

print(f"\nManipulation Detection Results:")
print(f"  Risk Level: {manipulation_result['manipulation_risk']}")
print(f"  Confidence: {manipulation_result['confidence']:.2f}")
print(f"  Risk Factors: {manipulation_result['risk_factors']}")
print(f"  Price Volatility: {manipulation_result['price_volatility']:.4f}")
print(f"  Oracle Consensus: {manipulation_result['oracle_consensus']:.2f}")
print(f"  Recommendation: {manipulation_result['recommendation']}")

## 8. Health Check and Monitoring

In [ ]:
# Test health check functionality
print("Oracle Health Check")
print("=" * 25)

health_status = oracle_manager.health_check()

print(f"Overall Status: {health_status['overall_status']}")
print(f"Active Oracles: {health_status['active_oracles']}/{health_status['total_oracles']}")
print(f"Timestamp: {health_status['timestamp']}")

print(f"\nIndividual Oracle Status:")
for oracle_name, status_info in health_status['oracle_status'].items():
    print(f"  {oracle_name}: {status_info['status']} "
          f"(reliability: {status_info['reliability']:.2f})")
    print(f"    URL: {status_info['url']}")

## 9. Configuration Flexibility Demo

In [ ]:
# Demonstrate configuration flexibility
print("Configuration Flexibility Demo")
print("=" * 35)

print(f"\nCurrent Configuration Highlights:")
print(f"  Aggregation Method: {oracle_manager.config.aggregation.method}")
print(f"  Confidence Threshold: {oracle_manager.config.validation.confidence_threshold}")
print(f"  Fallback Enabled: {oracle_manager.config.fallback.enabled}")
print(f"  Manipulation Detection: {oracle_manager.config.manipulation_detection.enabled}")

print(f"\nCircuit Breaker Settings:")
circuit = oracle_manager.config.circuit_breaker
print(f"  Enabled: {circuit.enabled}")
print(f"  Max Failures/Min: {circuit.max_failures_per_minute}")
print(f"  Recovery Time: {circuit.recovery_time_minutes} minutes")
print(f"  Escalation Thresholds: {circuit.escalation_thresholds}")

print(f"\nCaching Configuration:")
cache = oracle_manager.config.caching
print(f"  Enabled: {cache.enabled}")
print(f"  Default TTL: {cache.default_ttl_seconds}s")
print(f"  Max Cache Size: {cache.max_cache_size}")

print(f"\nMonitoring Settings:")
monitor = oracle_manager.config.monitoring
print(f"  Log Level: {monitor.log_level}")
print(f"  Health Check Interval: {monitor.health_check_interval_seconds}s")
print(f"  Metrics Enabled: {monitor.metrics_enabled}")
print(f"  Alert Thresholds: {monitor.alert_thresholds}")

## 10. MCP Service Configuration

In [ ]:
# Show MCP service integration configuration
print("MCP Service Integration Configuration")
print("=" * 45)

mcp_config = oracle_manager.config.mcp_services
print(f"\nConfigured MCP Services:")
print(f"  Algorand Reader URL: {mcp_config.algorand_reader_url}")
print(f"  Market Data URL: {mcp_config.market_data_url}")

# Test connectivity (mock)
print(f"\nMCP Service Integration Ready:")
print(f"  ✅ Algorand Reader service configured on port 8002")
print(f"  ✅ Market Data service configured on port 8003")
print(f"  ✅ Oracle engine can integrate with both services")

print(f"\nNote: In production, the oracle engine would:")
print(f"  - Fetch real-time price data from MCP services")
print(f"  - Validate prices against multiple sources")
print(f"  - Apply configuration-driven aggregation")
print(f"  - Provide reliable price feeds to the lending platform")

## Summary

This notebook has demonstrated the configuration-driven Oracle Price Integration Engine, showing:

✅ **Configuration Management**: YAML-based configuration with environment variable overrides  
✅ **Oracle Provider Management**: Configurable oracle providers with reliability scoring  
✅ **Price Aggregation**: Multiple aggregation methods with configurable weights  
✅ **Validation & Thresholds**: Configurable confidence and deviation thresholds  
✅ **Fallback Mechanisms**: Emergency price fallbacks when oracles fail  
✅ **Manipulation Detection**: Configurable risk assessment and detection  
✅ **Health Monitoring**: Oracle status tracking and circuit breakers  
✅ **Asset-Specific Config**: Per-asset refresh intervals and settings  
✅ **MCP Integration**: Ready for integration with MCP services on ports 8002/8003  

The oracle engine is now fully configurable and ready for production use with the Algorand lending ecosystem.